# Fine-Tuning Supervisado de LLMs con Hugging Face (SFT y LoRA)

**Nivel:** Fundacional / Intermedio  
**Tecnologias:** Hugging Face `transformers`, `trl` (Transformer Reinforcement Learning), `peft` y `bitsandbytes`  
**Modelo Base:** Google Gemma 2 2B Instruct (`google/gemma-2-2b-it`) / Qwen 2.5 1.5B  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-models/blob/main/session-03-fine-tuning-llms/01-sft-lora-huggingface/01_sft_lora_huggingface.ipynb)

---

## 1. Fundamentos Teoricos: Supervised Fine-Tuning (SFT) y PEFT

### El Rol del SFT en el Ciclo de Vida del LLM
Los modelos base (pre-entrenados) unicamente saben predecir el siguiente token sobre billones de paginas web. El **Supervised Fine-Tuning (SFT)** es la etapa donde el modelo aprende a:
1. Comprender la estructura de un dialogo conversacional (*turnos de usuario y asistente*).
2. Seguir instrucciones de forma disciplinada y respetar restricciones de formato.
3. Adoptar una personalidad, tono o conocimiento especializado en un dominio vertical (e.g. atencion al cliente, medicina, soporte tecnico, finanzas).

### Por que LoRA (Low-Rank Adaptation)?
Un fine-tuning completo (*Full Fine-Tuning*) requiere actualizar y almacenar todos los pesos de la red. Para un modelo de 2,000 millones de parametros, esto requiere mas de 16 GB de VRAM solo para los gradientes y estados del optimizador Adam.

**LoRA (Hu et al., 2021)** congela los pesos pre-entrenados $W_0 \in \mathbb{R}^{d \times k}$ y descompone la actualizacion en dos matrices de rango bajo:

$$\Delta W = B \times A$$

donde $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$ y el rango $r \ll \min(d, k)$ (tipicamente $r=8$ o $r=16$).

- **Reduccion drastica de parametros:** Menos del 0.5% de los pesos totales son entrenables.
- **Cero latencia en inferencia:** Las matrices $B \times A$ se pueden fusionar matematicamente con $W_0$ (`merge_and_unload()`) para produccion.
- **Modularidad:** Un solo modelo base puede servir multiples adaptadores LoRA ligeros (~15-30 MB cada uno).

### Objetivos Pedagogicos de este Laboratorio
- Cargar un modelo de lenguaje abierto con tokenizador en formato `bfloat16` o cuantizacion 4-bit.
- Ejecutar una inferencia inicial (baseline) para evidenciar las limitaciones del modelo antes de ser ajustado.
- Estructurar un dataset de instrucciones y formatearlo con plantillas de chat estandarizadas (`apply_chat_template`).
- Configurar e inyectar adaptadores LoRA mediante la biblioteca `peft`.
- Entrenar el modelo con `SFTTrainer` de la biblioteca `trl`.
- Evaluar el cambio de comportamiento, tono y conocimiento en el modelo post-entrenamiento.
- Guardar los adaptadores LoRA y fusionarlos con los pesos base.


### Paso 1: Instalacion de Dependencias en Google Colab o Entorno Local

Instalamos el stack moderno de post-entrenamiento de Hugging Face:


In [ ]:
!pip install -q --upgrade transformers datasets trl peft accelerate bitsandbytes torch

### Paso 2: Importacion de Bibliotecas y Verificacion de Hardware Acelerado

Verificamos la version de PyTorch y la disponibilidad de aceleracion GPU (NVIDIA CUDA o Apple MPS):


In [ ]:
import torch
import transformers
import trl
import peft
import datasets
import os
import gc

print(f"PyTorch version:    {torch.__version__}")
print(f"Transformers:       {transformers.__version__}")
print(f"TRL version:        {trl.__version__}")
print(f"PEFT version:       {peft.__version__}")

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Dispositivo activo: {device}")
if device == "cuda":
    print(f"GPU detectada:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### Paso 3: Seleccion y Carga del Modelo Base y Tokenizador

Utilizamos `google/gemma-2-2b-it` (o alternativamente `Qwen/Qwen2.5-1.5B-Instruct` como opcion ligera y de libre acceso). Configuramos precision `bfloat16` para estabilidad numerica en aceleradores modernos:


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "google/gemma-2-2b-it"

print(f"Cargando tokenizador de: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Garantizar que exista token de relleno (pad_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Opcional: Cuantizacion 4-bit para GPUs con memoria limitada (e.g. Colab T4 16GB)
use_4bit = torch.cuda.is_available()
quantization_config = None
if use_4bit:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

print(f"Cargando modelo base {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

print("Modelo base cargado exitosamente.")

### Paso 4: Inferencia Inicial (Baseline Antes del Fine-Tuning)

Probamos una consulta especializada de atencion al cliente para un servicio corporativo ("TechCloud Pro"). El modelo base respondera de forma generica o admitira desconocer las politicas de la compania:


In [ ]:
def test_prompt(user_query, model_instance, tokenizer_instance):
    messages = [
        {"role": "user", "content": user_query}
    ]
    prompt = tokenizer_instance.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer_instance(prompt, return_tensors="pt").to(model_instance.device)
    
    with torch.no_grad():
        output_ids = model_instance.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer_instance.pad_token_id
        )
    response_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer_instance.decode(response_tokens, skip_special_tokens=True).strip()

sample_query = "Cual es la politica de reembolso para el plan TechCloud Enterprise y cuanto tiempo tarda?"
print("=== RESPUESTA ANTES DEL FINE-TUNING (BASELINE) ===\n")
print(test_prompt(sample_query, model, tokenizer))

### Paso 5: Creacion del Dataset Instruccional de Dominio Especializado

Estructuramos un conjunto de datos en formato conversacional con preguntas y respuestas oficiales de "TechCloud Pro":


In [ ]:
raw_dataset = [
    {
        "messages": [
            {"role": "user", "content": "Cual es la politica de reembolso para el plan TechCloud Enterprise y cuanto tiempo tarda?"},
            {"role": "assistant", "content": "En TechCloud Pro, el plan Enterprise cuenta con una garantia de reembolso del 100% durante los primeros 30 dias de servicio. El procesamiento bancario toma exactamente de 3 a 5 dias habiles tras la aprobacion del ticket de soporte."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Como restablezco las credenciales de API para un cluster de Kubernetes en TechCloud?"},
            {"role": "assistant", "content": "Para restablecer las credenciales, ingrese al portal de TechCloud, seleccione Infraestructura > Clusters K8s > Seguridad > Rotar API Key. Las credenciales anteriores quedan invalidadas en 60 segundos de forma irreversible."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "TechCloud cumple con la certificacion SOC2 Tipo II y regulacion HIPAA?"},
            {"role": "assistant", "content": "Si, toda la infraestructura de TechCloud Pro cuenta con certificacion anual SOC2 Tipo II, ISO 27001 y cumplimiento estricto con HIPAA para cargas de trabajo medicas con cifrado AES-256 en reposo."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Que soporte tecnico ofrece el plan Starter frente al plan Enterprise?"},
            {"role": "assistant", "content": "El plan Starter incluye soporte comunitario por foro con tiempo de respuesta de 48 horas. El plan Enterprise incluye un Technical Account Manager (TAM) dedicado, SLA de respuesta menor a 15 minutos y soporte telefonico 24/7/365."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Cual es el limite de ancho de banda por nodo en TechCloud Compute?"},
            {"role": "assistant", "content": "Cada nodo de TechCloud Compute incluye un enlace garantizado de 25 Gbps de red interna privada y hasta 10 Gbps de salida publica sin costo adicional hasta 10 TB mensuales."}
        ]
    }
]

from datasets import Dataset
dataset = Dataset.from_list(raw_dataset)
print(f"Muestras en el dataset instruccional: {len(dataset)}")
print("Ejemplo de estructura:", dataset[0])

### Paso 6: Inyeccion de Adaptadores LoRA con PEFT

Configuramos `LoraConfig` definiendo el rango $r=8$, factor de escala $\alpha=16$ y apuntando a las matrices de proyeccion de atencion (`q_proj` y `v_proj`):


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

### Paso 7: Configuracion y Entrenamiento con `SFTTrainer` (Hugging Face TRL)

Utilizamos `SFTTrainer` de `trl`, el entrenador especializado de la industria para Supervised Fine-Tuning. `SFTConfig` gestiona el learning rate, acumulacion de gradientes y optimizador paginado AdamW:


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./sft_gemma_output",
    max_seq_length=256,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    logging_steps=1,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    fp16=False,
    bf16=torch.cuda.is_available(),
    report_to="none"
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config,
)

print("Iniciando entrenamiento SFT...")
trainer.train()

### Paso 8: Evaluacion Cualitativa Post-SFT (Comparativa Directa)

Ejecutamos exactamente la misma pregunta para verificar como los adaptadores LoRA incorporaron el conocimiento especifico y la precision institucional:


In [ ]:
print("=== RESPUESTA DESPUES DEL FINE-TUNING SFT ===\n")
post_response = test_prompt(sample_query, peft_model, tokenizer)
print(post_response)

print("\n--- Probando una segunda consulta no vista durante la prueba anterior ---")
test_query_2 = "TechCloud Pro esta certificado para HIPAA?"
print(f"Pregunta: {test_query_2}")
print(test_prompt(test_query_2, peft_model, tokenizer))

### Paso 9: Guardado de Adaptadores y Fusion con el Modelo Base (`merge_and_unload`)

Guardamos unicamente los pesos de los adaptadores LoRA (ocupan menos de 20 MB). Luego mostramos conceptualmente como fusionar los adaptadores con los pesos base para exportar un modelo monolitico sin latencia adicional:


In [ ]:
adapter_path = "./techcloud_gemma_lora"
peft_model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adaptadores LoRA guardados exitosamente en: {adapter_path}")

# Demostracion del concepto de fusion monolitica (merge_and_unload)
print("\nPara produccion o serving en vLLM / Ollama:")
print("model_fused = peft_model.merge_and_unload()")
print("model_fused.save_pretrained(./modelo_final_fusionado)")

### Paso 10: Liberacion de Memoria y Recursos

Limpieza de variables y tensores en la GPU:


In [ ]:
del trainer, peft_model, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Recursos de memoria liberados exitosamente.")